> `oeai_mod_bromcom_env_var.ipynb`
> [WIP]
> *Module configuration*

In [ ]:
%run ./oeai_py

In [ ]:
%run ./oeai_logger

In [ ]:
# Create an instance of OEAI class and set the platform ("Synapse" or "Fabric")
oeai = OEAI(platform="Fabric")

In [ ]:
# CHANGE VALUES FOR YOUR KEY VAULT
keyvault = "" # Fabric requires full URL eg "https://key_vault_name.vault.azure.net/"
keyvault_linked_service = "" # Not required for Fabric.

In [ ]:
# Bromcom connection - REQUIRED
connection = "sql" # "sql" | "odata"

In [ ]:
# INITIALISE LOGGING
oeai.log = OEAILogger()
oeai.module_id = f"bromcom_{connection}"
oeai.client_id = oeai.get_secret(spark, "oeai-api-id", keyvault_linked_service, keyvault)

# SET HANDLER - Console Output
oeai.log.handler.stream = SimpleNamespace(
    level=_lib_logging.INFO, 
    formatter="stream"
)

# SET HANDLER - HTTP/API
oeai.log.handler.http = SimpleNamespace(
    level = _lib_logging.INFO,
    formatter = "json",
    api_base_url = oeai.get_secret(spark, "oeai-api-url", keyvault_linked_service, keyvault),
    api_key = oeai.get_secret(spark, "oeai-api-key", keyvault_linked_service, keyvault), 
    urgent_event_level = _lib_logging.ERROR,
    urgent_event_types = {"block.start"},
    flush_batch_size = 50,
    flush_interval_s = 10,
    flush_on_urgent = True,
    background_flush = True,
    # timeout_s = 3.0,
    # print_response = True,
)

In [ ]:
# OEA environment paths
import os
from types import SimpleNamespace
storage_root = oeai.get_secret(spark, "storage-root", keyvault_linked_service, keyvault)
oeai.path = SimpleNamespace(
    reference = os.path.join(storage_root, "reference/"),
    bronze    = os.path.join(storage_root, f"oeai_bronze/{oeai.module_id}/"),
    silver    = os.path.join(storage_root, f"oeai_silver/"),
    gold      = os.path.join(storage_root, f"oeai_gold/"),
)

bronze_path     = oeai.path.bronze
silver_path     = oeai.path.silver
gold_path       = oeai.path.gold
reference_path  = oeai.path.reference

In [ ]:
print(f"Loading Env Var for Bromcom via {connection}")

In [ ]:
def get_reference_path(gold_path: str, file_name: str) -> str:
    try:
        # If oeai.path.reference exists and is valid, use it
        ref_base = oeai.path.reference
        if not ref_base:  # catch empty string or None
            raise AttributeError
    except (AttributeError, NameError):
        # Derive from gold_path by replacing the last folder with "reference"
        parent = os.path.dirname(gold_path.rstrip("/"))
        ref_base = os.path.join(parent, "reference")

    return os.path.join(ref_base, file_name)

In [ ]:
###
# Secrets for connection
###

# Logic block based on connection type
if connection == "sql":
    print("Setting up SQL secrets and database config...")
    
    jdbc_url = (
        f"jdbc:sqlserver://{oeai.get_secret(spark, 'bromcom-sql-hostname', keyvault_linked_service, keyvault)}:"
        f"{oeai.get_secret(spark, 'bromcom-sql-port', keyvault_linked_service, keyvault) or 1433};"
        f"database={oeai.get_secret(spark, 'bromcom-sql-database', keyvault_linked_service, keyvault)}"
    )
    
    db_properties = {
        "user":     oeai.get_secret(spark, "bromcom-sql-username", keyvault_linked_service, keyvault),
        "password": oeai.get_secret(spark, "bromcom-sql-password", keyvault_linked_service, keyvault),
        "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    }
elif connection == "odata":
    print("Setting up OData secrets...")
    bromcom_odata_feed_password = oeai.get_secret(spark, "bromcom-odata-password", keyvault_linked_service, keyvault)
    bromcom_odata_feed_username = oeai.get_secret(spark, "bromcom-odata-username", keyvault_linked_service, keyvault)
else:
    # Raise an error if connection type is not recognized
    raise ValueError("Connection type not set or invalid. Must be 'sql' or 'odata'.")

In [ ]:
###
# Bronze config
###

# Logic block based on connection type
if connection == "sql":
    print("Setting up Bronze config with selected views in SQL database ...")
    selected_views = [
        "dbo_vwVisionSchools",
        "dbo_vwStudents",
        "Student_vwSessionAttendance",
        "Student_vwExclusions",
        "Student_vwBehaviourEvents",
        "dbo_vwStaff",
        "Staff_vwRoles",
        "Staff_vwAbsences",
        "Student_vwExamResults",
        # "Student_vwEbaccResults",
        # "Student_vwCurriculumAssessmentResults",
        "Student_vwCTFAssessments",
        # "Student_vwAssessmentResults",
    ]

elif connection == "odata":
    print("Setting up Bronze config with selected entities and date parameter...")

    # Map source entity -> desired output name (folder/table name under bronze_path)
    entity_mappings = {
        "Students": "dbo_vwStudents",
        "VisionSchools": "dbo_vwVisionSchools",
        "Exclusions": "Student_vwExclusions",
        "BehaviourEvents": "Student_vwBehaviourEvents",
        "SessionAttendance": "Student_vwSessionAttendance",
        "Staff": "dbo_vwStaff",
        "StaffRoles": "Staff_vwRoles",
        "StaffAbsences": "Staff_vwAbsences",
        "ExamResults": "Student_vwExamResults",
        "EbaccResults": "Student_vwEbaccResults",
        "CurriculumAssessmentResults": "Student_vwCurriculumAssessmentResults",
        "CTFAssesments": "Student_vwCTFAssessments", # "Assessment" is misspelt as the odata endpoint = "CTFAssesments"[sic]
        "AssessmentResults": "Student_vwAssessmentResults",
    }

    # --- Entities where we pull EVERYTHING:
    FULL_ENTITIES = [
        "Students",
        "VisionSchools",
        "Exclusions",
        "Staff",
        "StaffRoles",
        "StaffAbsences",
        "ExamResults",
        # "EbaccResults",
        # "CurriculumAssessmentResults",
        # "CTFAssesments", # Use this to get all on first run
    ]
    
    FILTERED_ENTITIES = {
        "BehaviourEvents":   {"date_col": "EventDateTime",  "mode": "weekly_range"},
        "SessionAttendance": {"date_col": "AttendanceDate", "mode": "weekly_upper_bound"},
        # "AssessmentResults":   {"date_col": "ResultDate",  "mode": "weekly_range"},
        "CTFAssesments":   {"date_col": "ResultDate",  "mode": "weekly_range"}, # Use this daily runs

    }

    # --- Set the earliest date to pull (UTC). We'll walk back from "tomorrow 00:00Z" to this date (inclusive).
    from datetime import datetime

    def get_bound_start_date(manual_date: datetime | None = None) -> datetime:
        """
        Returns the most recent August 1st unless a manual date is provided.
        """
        if manual_date:
            return manual_date  # manual override

        now = datetime.now()
        # If we're before August 1 this year, use last year's August 1
        year = now.year if now.month >= 8 else now.year - 1
        return datetime(year, 8, 1, 0, 0, 0)


    # Example usage:
    BOUND_START_DATE = get_bound_start_date()  # auto sets to most recent Aug 1
    # or manually override:
    # BOUND_START_DATE = get_bound_start_date(datetime(2025, 8, 1))

else:
    # Raise an error if connection type is not recognized
    raise ValueError("Connection type not set or invalid. Must be 'sql' or 'odata'.")

In [ ]:
###
# Silver mapping from views in Bronze
###
print("Setting up Silver delta table mappings ...")
delta_table_name_mapping = {
    "dbo_vwVisionSchools": ["dim_Organisation"],
    "dbo_vwStudents": ["dim_Student", "dim_StudentExtended"],
    "Student_vwSessionAttendance": ["fact_AttendanceSession"],
    "Student_vwExclusions": ["fact_Exclusion", "dim_ExclusionReason"],
    "Student_vwBehaviourEvents": ["fact_Behaviour", "fact_Achievement"],
    "dbo_vwStaff": ["dim_Staff"],
    "Staff_vwRoles": ["dim_StaffRole"],
    "Staff_vwAbsences": ["fact_StaffAbsence"],
    # "Student_vwExamResults": ["fact_ExamResult"],
}

In [ ]:
###
# Academic Year Config
###

ACADEMIC_YEAR_START = {"month": 8, "day": 21}

print(f"Academic Year Starts on: {ACADEMIC_YEAR_START['month']:02d}/{ACADEMIC_YEAR_START['day']:02d}")